In [1]:
import scipy.io as sio
import pandas as pd
import numpy as np
import sys, os
import torch
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from data_classes.decomposition import Extract_Features

In [2]:
torch.manual_seed(42)
device = torch.device(
                    "cuda"
                    if torch.cuda.is_available()
                    else "mps" if torch.backends.mps.is_available() else "cpu"
                )
print(f"Using {device}")

# load & preprocess
data = sio.loadmat('../data/mine_impact_data_2019.mat')
samps = pd.DataFrame(data['x'].T)
labs  = pd.DataFrame(data['y'].T, columns=['y'])
df = pd.concat([samps, labs], axis=1).dropna().sample(frac=1, random_state=42)


Using cuda


In [3]:
shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_X = shuffled_df.iloc[:, :-1]
df_Y = shuffled_df.iloc[:, -1]

data = Extract_Features(df_X, df_Y, feature="raw")
print(data.get_samples().shape, data.get_labels().shape)

(3309, 36000) (3309,)


In [4]:
import models.classification as classify
import models.loops as loops
import models.models as models

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = models.Convolution()

loops.train(model=model, model_path="./model_paths/convolution.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-3, optim="adam", epochs=20)

loops.test(model_path="./model_paths/convolution.pth", test_loader=test_loader, report=True)

[INFO] EPOCH: 1/20
Train loss: 0.644887, Train accuracy: 0.6637
[INFO] EPOCH: 2/20
Train loss: 0.624295, Train accuracy: 0.6637
[INFO] EPOCH: 3/20
Train loss: 0.538241, Train accuracy: 0.6723
[INFO] EPOCH: 4/20
Train loss: 0.436085, Train accuracy: 0.8277
[INFO] EPOCH: 5/20
Train loss: 0.354579, Train accuracy: 0.8677
[INFO] EPOCH: 6/20
Train loss: 0.310047, Train accuracy: 0.8770
[INFO] EPOCH: 7/20
Train loss: 0.280077, Train accuracy: 0.8870
[INFO] EPOCH: 8/20
Train loss: 0.251827, Train accuracy: 0.8977
[INFO] EPOCH: 9/20
Train loss: 0.217307, Train accuracy: 0.9070
[INFO] EPOCH: 10/20
Train loss: 0.210751, Train accuracy: 0.9113
[INFO] EPOCH: 11/20
Train loss: 0.191149, Train accuracy: 0.9137
[INFO] EPOCH: 12/20
Train loss: 0.169866, Train accuracy: 0.9213
[INFO] EPOCH: 13/20
Train loss: 0.152220, Train accuracy: 0.9327
[INFO] EPOCH: 14/20
Train loss: 0.149875, Train accuracy: 0.9433
[INFO] EPOCH: 15/20
Train loss: 0.135251, Train accuracy: 0.9477
[INFO] EPOCH: 16/20
Train loss: 0.